## DSA 4060: Week 2 
### Recommendation Data and Popularity Baselines
### Objective
Clean and explore movie-rating data, calculate user-item matrix sparsity, and build a popularity-based recommender that ranks movies by average rating and number of ratings.
### 1. Import Libraries
Use pandas for data manipulation and NumPy for numerical computations.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

### 2. Load and Inspect the Dataset

The dataset contains movie-rating interactions, including user IDs, movie IDs, ratings, and timestamps. I will inspect the first ten records, dataset dimensions, data types, missing values, and duplicate records to understand the data before cleaning it.

In [6]:
# Define the dataset filename
DATA_FILE = Path("DSA_4060_Week2_Movie_Ratings.csv")

# Check that the file exists
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE.resolve()}"
    )

# Load the dataset
df = pd.read_csv(DATA_FILE)

# Standardise column names
df.columns = df.columns.str.strip().str.lower()

# Support common alternative column names
df = df.rename(columns={
    "userid": "user_id",
    "movieid": "movie_id"
})

# Confirm the required columns exist
required_columns = [
    "user_id", "movie_id", "rating", "timestamp"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {df.columns.tolist()}"
    )

print("Dataset loaded successfully!")
print("File:", DATA_FILE.resolve())

# Display the first 10 records
print("FIRST 10 RECORDS OF THE MOVIE RATINGS DATASET")
display(df.head(10))

# Dataset dimensions
print("\nDataset dimensions:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Data types
print("\nData types:")
print(df.dtypes)

# Missing values
print("\nMissing values:")
print(df.isnull().sum())

# Duplicate records
print("\nDuplicate records:", df.duplicated().sum())

Dataset loaded successfully!
File: C:\Users\ADMIN\Downloads\DSA_4060_Week2\DSA_4060_Week2_Movie_Ratings.csv
FIRST 10 RECORDS OF THE MOVIE RATINGS DATASET


,user_id,movie_id,movie_title,rating,timestamp
0,U001,M001,Nairobi Nights,3,2026-01-01T08:00:00Z
1,U004,M001,Nairobi Nights,4,2026-01-06T09:00:00Z
2,U007,M001,Nairobi Nights,2,2026-01-11T10:00:00Z
3,U010,M001,Nairobi Nights,2,2026-01-16T11:00:00Z
4,U013,M001,Nairobi Nights,4,2026-01-21T12:00:00Z
5,U016,M001,Nairobi Nights,3,2026-01-26T13:00:00Z
6,U019,M001,Nairobi Nights,3,2026-01-31T14:00:00Z
7,U022,M001,Nairobi Nights,2,2026-02-05T15:00:00Z
8,U025,M001,Nairobi Nights,2,2026-02-10T16:00:00Z
9,U028,M001,Nairobi Nights,4,2026-02-15T17:00:00Z



Dataset dimensions: (509, 5)
Number of rows: 509
Number of columns: 5

Data types:
user_id        object
movie_id       object
movie_title    object
rating          int64
timestamp      object
dtype: object

Missing values:
user_id        2
movie_id       1
movie_title    1
rating         0
timestamp      0
dtype: int64

Duplicate records: 3


### 3. Data Cleaning

I will remove duplicate records and records with missing user IDs or movie IDs. I will convert ratings to numeric values, identify the rating scale, and remove invalid ratings based on the dataset's expected range.

In [10]:
# Record the original number of rows
original_rows = len(df)

# Remove duplicate records
df = df.drop_duplicates()

# Remove rows with missing user or movie IDs
df = df.dropna(subset=["user_id", "movie_id"])

# Remove blank IDs if IDs are stored as text
for col in ["user_id", "movie_id"]:
    df = df[
        df[col].astype("string").str.strip().ne("")
    ]

# Convert ratings to numeric; invalid text becomes NaN
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Remove missing ratings
df = df.dropna(subset=["rating"])

# Inspect the rating scale before validating
print("Minimum rating:", df["rating"].min())
print("Maximum rating:", df["rating"].max())
print("Unique rating values:", sorted(df["rating"].unique()))

# Standard MovieLens-compatible range: 0.5 to 5.0
# Confirm the actual scale from the lecturer's dataset.
df = df[df["rating"].between(0.5, 5.0)]

# Final checks
print("\nRows before cleaning:", original_rows)
print("Rows after cleaning:", len(df))
print("Rows removed:", original_rows - len(df))
print("Missing user IDs:", df["user_id"].isna().sum())
print("Missing movie IDs:", df["movie_id"].isna().sum())
print("Duplicate records remaining:", df.duplicated().sum())
print("Final rating range:", df["rating"].min(), "to", df["rating"].max())

Minimum rating: 1
Maximum rating: 5
Unique rating values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Rows before cleaning: 500
Rows after cleaning: 500
Rows removed: 0
Missing user IDs: 0
Missing movie IDs: 0
Duplicate records remaining: 0
Final rating range: 1 to 5


#### Cleaning Results
After removing duplicates, missing user/movie IDs, and ratings outside the valid 1–5 range, the dataset contains 500 valid ratings. This cleaned dataset is used for further analysis to ensure accurate and reliable recommendations.

### 4. Exploratory Data Analysis

I will analyse the cleaned dataset to determine the total number of ratings, unique users, unique movies, most active user, most-rated movie, and overall average rating. These statistics help describe user activity and movie popularity.

In [11]:
# Basic interaction statistics
total_ratings = len(df)
unique_users = df["user_id"].nunique()
unique_movies = df["movie_id"].nunique()
average_rating = df["rating"].mean()

# Most active user
user_activity = df["user_id"].value_counts()
most_active_user = user_activity.index[0]
most_active_user_count = user_activity.iloc[0]

# Most-rated movie
movie_activity = df["movie_id"].value_counts()
most_rated_movie = movie_activity.index[0]
most_rated_movie_count = movie_activity.iloc[0]

print("Total ratings:", total_ratings)
print("Number of unique users:", unique_users)
print("Number of unique movies:", unique_movies)
print("Most active user:", most_active_user)
print("Ratings by most active user:", most_active_user_count)
print("Most-rated movie ID:", most_rated_movie)
print("Number of ratings for most-rated movie:", most_rated_movie_count)
print("Average rating:", round(average_rating, 2))

Total ratings: 500
Number of unique users: 50
Number of unique movies: 24
Most active user: U010
Ratings by most active user: 17
Most-rated movie ID: M001
Number of ratings for most-rated movie: 37
Average rating: 3.45


Analysis Summary:
- Total ratings: The number of rating records available after cleaning is 500
- Unique users: The number of distinct users who rated movies is 50
- Unique movies: The number of distinct movies appearing in the dataset is 24
- Most active user: The user who submitted the most ratings is U010
- Most-rated movie: The movie that received the greatest number of ratings is M005
- Average rating: The overall mean rating across all records is 3.5.

### 5. User-Item Interaction Matrix

A user-item interaction matrix represents users as rows and movies as columns. Each cell contains the user's rating for a movie. Missing cells indicate that the user has not rated the movie. If multiple ratings exist for the same user-movie pair, their average will be used.

In [12]:
# Create the user-item interaction matrix
user_item_matrix = df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating",
    aggfunc="mean"
)

# Display a small section of the matrix
display(user_item_matrix.iloc[:10, :10])

print("Matrix dimensions:", user_item_matrix.shape)

movie_id,M001,M002,M003,M004,M005,M006,M007,M008,M009,M010
user_id,,,,,,,,,,
U001,3.000000,2.000000,NaN,NaN,4.0,4.0,NaN,NaN,NaN,NaN
U002,3.000000,NaN,NaN,3.5,NaN,NaN,3.0,NaN,NaN,NaN
U003,NaN,NaN,NaN,NaN,4.0,NaN,NaN,4.0,4.0,NaN
U004,4.000000,NaN,NaN,NaN,3.0,3.0,NaN,NaN,NaN,NaN
U005,2.000000,4.000000,NaN,3.0,NaN,NaN,5.0,NaN,NaN,NaN
U006,3.000000,3.000000,NaN,5.0,4.0,NaN,NaN,3.0,NaN,NaN
U007,2.333333,NaN,NaN,NaN,5.0,4.0,NaN,NaN,2.0,NaN
U008,3.000000,2.000000,NaN,NaN,3.0,4.0,5.0,NaN,NaN,NaN
U009,NaN,NaN,NaN,4.0,4.0,3.0,NaN,3.5,4.0,NaN


Matrix dimensions: (50, 24)


A table with users as rows and movie IDs as columns. Some cells will contain ratings, while others will show Misssing value.

Using pivot_table() with aggfunc="mean" prevents errors if a user has rated the same movie more than once.

### 6. Calculate Matrix Sparsity

Sparsity measures the proportion of user-movie combinations for which no rating exists. A highly sparse matrix indicates that users have rated only a small fraction of all available movies.


In [13]:
# Count unique user-movie pairs with ratings
observed_interactions = df[
    ["user_id", "movie_id"]
].drop_duplicates().shape[0]

number_of_users = df["user_id"].nunique()
number_of_movies = df["movie_id"].nunique()

total_possible_interactions = number_of_users * number_of_movies

sparsity = 1 - (
    observed_interactions / total_possible_interactions
)

print("Observed user-movie interactions:", observed_interactions)
print("Total possible interactions:", total_possible_interactions)
print(f"Sparsity: {sparsity:.4%}")
print(f"Matrix density: {(1 - sparsity):.4%}")

Observed user-movie interactions: 430
Total possible interactions: 1200
Sparsity: 64.1667%
Matrix density: 35.8333%


#### Interpretation of Sparsity

The dataset contains 500 observed ratings out of 1,200 possible user–movie interactions, resulting in a sparsity level of 64.1667%. This means that 64.1667% of the user–item matrix is empty, while 35.8333% contains recorded ratings.

This sparsity presents a challenge for recommender systems, which must generate useful recommendations despite limited user–movie interaction data.

### 7. Popularity-Based Recommendation

A popularity-based recommender ranks movies using their average rating and number of ratings. To reduce the effect of movies with very few ratings, I will include only movies with at least 10 ratings. The remaining movies will be ranked by average rating, with rating count used to break ties.

In [14]:
# Calculate average rating and number of ratings for each movie
movie_stats = df.groupby("movie_id").agg(
    average_rating=("rating", "mean"),
    number_of_ratings=("rating", "count")
).reset_index()

# Keep movies with at least 10 ratings
eligible_movies = movie_stats[
    movie_stats["number_of_ratings"] >= 10
].copy()

# Rank by average rating, then number of ratings
top_10_movies = eligible_movies.sort_values(
    by=["average_rating", "number_of_ratings", "movie_id"],
    ascending=[False, False, True]
).head(10).copy()

# Format the average rating
top_10_movies["average_rating"] = (
    top_10_movies["average_rating"].round(2)
)

# Display the recommendations
display(top_10_movies)

print("Number of eligible movies:", len(eligible_movies))
print("Number of recommendations:", len(top_10_movies))

,movie_id,average_rating,number_of_ratings
20,M021,4.47,15
12,M013,4.35,17
13,M014,4.25,16
18,M019,4.13,15
19,M020,4.13,15
6,M007,4.12,16
5,M006,3.91,34
4,M005,3.86,37
11,M012,3.76,17
3,M004,3.67,27


Number of eligible movies: 24
Number of recommendations: 10


The dataset contained 24 eligible movies that received at least 10 ratings. From these, the popularity-based recommender selected the top 10 movies based on their average ratings, using the number of ratings to break ties. These movies represent the highest-rated options among those meeting the minimum rating requirement.

### 8. Interpretation of the Recommendations

The recommended movies rank highest because they have relatively high average ratings and at least 10 user ratings. This minimum threshold helps ensure that recommendations are based on sufficient feedback rather than a single high rating.

### Advantages

The popularity-based recommender is simple, fast, easy to implement, and straightforward to understand. It is also useful when users’ individual preferences are not yet available.

### Limitations

1. **Lack of personalization:** All users receive similar recommendations, regardless of their individual interests and preferences.
2. **Popularity bias:** Popular movies receive more visibility, while less-known movies may be overlooked. This limits recommendation diversity and reduces opportunities for lesser-known movies to reach new audiences.

### 9. Export Recommendations

In [15]:
# Create an output folder
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Save the recommendations
output_file = output_dir / "top_10_recommendations.csv"

top_10_movies.to_csv(output_file, index=False)

print("Recommendations saved to:", output_file.resolve())

Recommendations saved to: C:\Users\ADMIN\Downloads\DSA_4060_Week2\outputs\top_10_recommendations.csv
